# Introduction
In this notebook we show how to fetch and pre-process Sentinel-2 data for your area of interest.

<div class="alert alert-block alert-warning">
<b>Running this notebook on CDSE notebooks?</b><br>

Make sure you select the openeo python kernel.<br>

Execute the following cell to install some required python packages.

</div>

In [ ]:
!pip install ipykernel ipyleaflet geopandas geojson --quiet

## Step 1: Specify your area of interest

(please always use a small area for testing purposes to reduce download times)

Option 1: Use an existing shapefile/geopackage file.

Make sure your file is located in the same folder as this notebook and provide the full filename (including extension) by replacing the `your_filename` part in the code.

In [ ]:
from pathlib import Path
import geopandas as gpd

in_dir = Path('./')
infile = in_dir / 'your_filename'
gdf = gpd.read_file(infile)
gdf

Option 2: use the ui_map tool to draw your area of interest:

In [ ]:
# Use the interactive widget to draw a polygon on the map.
# Once you drew a polygon, enter a name for the polygon in the text box below the map.
# Then click the "submit" button to save the polygon to the database.

from vito_agri_tutorials.utils.map import ui_map
map = ui_map(geometry_type='polygon')

In [ ]:
gdf = map.get_objects()
gdf

In [ ]:
# Convert geodataframe into geojson object (expected by OpenEO)
import geojson

def gdf_to_geojson(gdf):
    """Convert a geopandas dataframe to a geojson object,
    which can be used as spatial_extent in OpenEO requests."""
    json = gdf.to_json()
    return geojson.loads(json)

spatial_extent = gdf_to_geojson(gdf)

## Step 2: Define your period of interest

In [ ]:
start_date = '2021-04-01'
end_date = '2021-07-31'

# define the temporal extent for the Sentinel-2 data
temp_ext_s2 = [start_date, end_date]

## Step 3: Define what needs to be done

In [ ]:
import openeo
from openeo.extra.spectral_indices.spectral_indices import compute_and_rescale_indices
from vito_agri_tutorials.openeo.auth import connect_openeo

# properties to filter the data
# Any acquistion with a cloud percentage higher than 80% is filtered out
# (this is a common threshold for Sentinel-2 data)
props = {"eo:cloud_cover": lambda v: v <= 80}

# Define the bands of interest
bands = ["B03", "B04", "B05", "B06", "B07", "B08", "B11", "B12"]

# Define the spectral indices to compute and their input/output ranges
idx_list = ["NDVI", "NDMI", "NDRE1", "NDRE2", "NDRE5"]
index_dict = {idx: [-1, 1] for idx in idx_list}
final_index_dict = {
        "collection": {"input_range": [0, 8000], "output_range": [0, 30000]},
        "indices": {
            index: {"input_range": index_dict[index], "output_range": [0, 30000]}
            for index in index_dict
        },
    }

# establish a connection to CDSE
c = connect_openeo(backend="cdse")

# load the scene classification layer from sentinel-2 and prepare the cloud mask
scl = c.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=temp_ext_s2, bands="SCL", properties=props
)

# Compute the cloud mask from the SCL layer
cloud_mask = scl.process(
        "to_scl_dilation_mask",
        data=scl,
        kernel1_size=17, kernel2_size=77,
        mask1_values=[2, 4, 5, 6, 7],
        mask2_values=[3, 8, 9, 10, 11],
        erosion_kernel_size=3
    )

# load the Sentinel-2 collection with the specified bands and properties
s2 = c.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=temp_ext_s2, bands=bands, properties=props
)
# apply cloud masking to filter out cloudy pixels
s2 = s2.mask(cloud_mask)

# compute the spectral indices
indices = compute_and_rescale_indices(s2, index_dict=final_index_dict)

# combine the original bands with the computed indices
combined = s2.merge_cubes(indices)

# filter the combined collection to include the desired bands and indices
result_indices = combined.filter_bands(bands + idx_list)

# aggregate the filtered collection to 10-daily intervals using the specified reducer
idx_dekad = result_indices.aggregate_temporal_period("dekad", reducer="median")

# apply linear interpolation to fill any temporal gaps
interpolated = idx_dekad.apply_dimension(
    dimension="t", process="array_interpolate_linear"
)

# finally, limit spatial extent to the area of interest
final_result = interpolated.filter_spatial(spatial_extent)

## Step 3: Run the processing and fetch the result:

Use the `execute_batch` function to launch processing. After completion, the result will be automatically downloaded to your current directory. You can also manually fetch the result from the [openeo web editor](https://openeo.dataspace.copernicus.eu/)

In [ ]:
outname = input('Provide a name for the output file WITHOUT extension:')

job = final_result.execute_batch(
    f'{outname}.nc', title="Sentinel-2 download"
)

Learn how to combine Sentinel-2 with Sentinel-1 data here:
https://github.com/Open-EO/openeo-community-examples/blob/742f36061831b02e08b557cd3d06352b5c95fcf8/python/BasicSentinelMerge/sentinel_merge.ipynb

## Step 4: Read the result

In [ ]:
from pathlib import Path
import xarray as xr

outfile = Path('./') / f'{outname}.nc'
ds = xr.open_dataset(outfile)
ds

In [ ]:
# Plot the NDVI
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ndvi = ds['NDVI'].values[:, 10, 10]
time = ds.coords['t'].values
ax.plot(time, ndvi, '-o')
plt.xticks(rotation=45, ha='right')
plt.title('NDVI')
plt.show()